# TorchTitan-NPU 的 VarLen+FSDP、VarLen+CP 与 Profiling

第 7 章已经介绍 FSDP、TP 与 CP 的集合通信。本章把 FSDP 和 CP 放回真实的 Qwen3-1.7B Wordle SFT，追踪 **VarLen attention 与 CP 的交互**：VarLen 需要知道每个隔离区间从哪里开始、到哪里结束，CP 用 AllToAll 交换 sequence/head 布局；两个 CP rank 必须使用同一份全局区间信息。真实样本各占一个区间，末尾 padding 若存在也会单独占一个区间。

FSDP/CP 可以形成等物理 slot 的对比：两组使用相同模型、TND VarLen backend、两卡参数分片和每步 16,384 个容器位置。不过 4096 与 8192 的装箱空间不同，每步装入的原始样本、有效 token 和 Attention 区间形状也可能不同；因此它不是只替换并行策略的纯微基准，必须先重放 DataLoader 再解释性能。

---


## 教程进度回顾

| 章节 | 内容 | 状态 |
|---|---|---|
| 第 1 章 | SFT 概念 + Wordle 任务 | ✅ 已完成 |
| 第 2 章 | TorchTitan 框架 + 环境配置 | ✅ 已完成 |
| 第 3 章 | 数据准备 + 基线训练 + 推理评测 | ✅ 已完成 |
| 第 4 章 | 融合算子优化 + Profiling | ✅ 已完成 |
| 第 5 章 | Attention 公式、算子与 TorchTitan dispatch | ✅ 已完成 |
| 第 6 章 | Sequence Packing、VarLen Attention 与端到端效率 | ✅ 已完成 |
| 第 7 章 | FSDP、TP、CP 集合通信与通信量分析 | ✅ 已完成 |
| **第 8 章** | **VarLen+FSDP、VarLen+CP 与端到端 Profiling** | ← 当前 |

---


## 本章目标

完成本章后，你将能够：

- 解释 VarLen 的隔离区间信息与 Ulysses CP sequence/head 重排之间的约束；
- 沿 TorchTitan-NPU 的实现路径定位 VarLen+CP backend、pre/post hook 与 AllToAll；
- 为 VarLen+FSDP 和 VarLen+CP 设计等容器 token 的受控 profiling；
- 用配置、通信路径和 normalized throughput 三层证据限定性能结论。

---


## 前置条件

- 完成第 6 章，理解 sequence packing、`cu_seq` 与 NPU TND VarLen Attention；
- 完成第 7 章，理解 FSDP 参数通信和 CP AllToAll 的数据布局变化；
- 能够运行 Qwen3-1.7B Wordle SFT recipe，并读取 TorchTitan 与 TorchTitan-NPU 源码；
- 当前环境可使用两张 Ascend NPU 运行训练和 profiler。

---


## 本章结构

| Notebook | 内容 |
|---|---|
| [08.01](08.01_chapter_intro.ipynb) | 章节介绍（本节） |
| [08.02](08.02_varlen_cp_interaction.ipynb) | CP 8192 的配置动机、greedy packing 等价性、通信账本与 NPU 实现 |
| [08.03](08.03_torchtitan_profiling.ipynb) | VarLen+FSDP 4096 与 VarLen+CP 8192 的等 token profiling |
| [08.04](08.04_profiling_analysis.ipynb) | trace/CSV 解析、通信账本、差异归因和结论边界 |
| [08.05](08.05_chapter_practice.ipynb) | 按 08.01–08.04 分组的判断题与选择题 |

---


## 两条对比路线

| 路线 | parallelism | attention backend | 关键通信 |
|---|---|---|---|
| VarLen+FSDP | `dp_replicate=1, dp_shard=2, cp=1` | NPU TND VarlenAttention | parameter all-gather / gradient reduce-scatter |
| VarLen+CP | `dp_replicate=1, dp_shard=1, cp=2` | NPU TND VarlenAttention | Q/K/V 与 output all-to-all；CP mesh 同时触发参数管理 |

当前 TorchTitan 上游的 Qwen3 `update_from_config()` 会拒绝 VarLen+CP；TorchTitan-NPU 仓库通过 `NPUVarlenAttention`、`NPUVarlenUlyssesCP` 和 `sft_qwen3_1_7b_wordle_tnd` 支持这条路线。08.02 会定位这些仓库内实现。

---


## 固定实验卡与公平性

- Qwen3-1.7B、Wordle parquet、bf16、两张 Ascend NPU；
- `local_batch_size=2`；VarLen+FSDP 使用 `seq_len=4096, GBS=4`，VarLen+CP 使用 `seq_len=8192, GBS=2`；
- 两组每个 optimizer step 均处理 `16384` 个容器 token，每卡 microbatch 均为 `8192` token；
- profile 同时采集两个 rank 的 Step 5（`profile_ranks=-1, profile_step_start=5, profile_step_end=6`）；
- 吞吐关闭 profiler，并按 raw samples、input tokens、supervised tokens 归一化。

profile step 比较等容器 token 工作量；raw sample 和 supervised token 仍需从 DataLoader 计数。


In [ ]:
EXPERIMENT_CARD = {
    'model': 'Qwen3-1.7B',
    'dataset': 'assets/data/wordle',
    'dtype': 'bf16',
    'world_size': 2,
    'seq_len': {'varlen_fsdp': 4096, 'varlen_cp': 8192},
    'local_batch_size': 2,
    'profile_step': 5,
    'notebooks': ('08.02 VarLen+CP', '08.03 profiling', '08.04 analysis', '08.05 practice'),
}
for key, value in EXPERIMENT_CARD.items():
    print(f'{key:>18}: {value}')


---

## 本章产物与验收

1. DataLoader 让每条样本的位置编号从 0 重新开始，VarLen+CP smoke 能保留由此生成的完整 `cu_seq_q/cu_seq_k`，并在 CP pre/post hook 中看到 AllToAll；
2. VarLen+CP 的 TND kernel 使用 `actual_seq_qlen/actual_seq_kvlen` 和 `sparse_mode=7`；
3. Profiling 结果只使用当前 trace 的 CSV/JSON，不填入历史或猜测的 kernel 数字；
4. 最终性能结论同时有配置、通信路径和 normalized throughput 三层证据。

本轮 `_positions_v2` 结果已经通过这些门槛。结论不是笼统的“CP 更快”：CP+8192 的 median step 比 FSDP+4096 慢 5.3%，但由于 padding 更少，non-padding token/s、supervised token/s 和 raw sample/s 分别高 11.6%、8.8% 和 24.1%。第 8 章将把“物理 step 速度”和“业务有效数据吞吐”分开解释。


## 练习

1. （判断题）CP ranks 共同处理同一批上下文，计算 global batch size 时不能把 CP degree 当作数据并行副本数。

2. （判断题）两组实验处理相同数量的容器位置，就能断定 raw sample、non-padding token 和 supervised token 完全相同。

3. （单选题）本章的主要对比对象是什么？
    A. VarLen+FSDP 4096 与 VarLen+CP 8192
    B. DDP 与 Pipeline Parallel
    C. Dense 推理与量化推理
    D. TP 与 Expert Parallel

4. （多选题）形成最终性能结论需要哪些层面的证据？
    A. resolved config 与输入工作量
    B. 双 rank trace 中的 kernel 和 collective
    C. raw sample、有效 token 与监督 token 吞吐
    D. 只看一个最快 Attention kernel

In [ ]:
!cat ./answer/08.01_answer.txt
